# 4.3 Classification Models — Concepts & Examples

## 📋 Topics Covered

| # | Model | Key Idea |
|---|-------|----------|
| 1 | Logistic Regression | Linear decision boundary via sigmoid function |
| 2 | K-Nearest Neighbors | Vote from k closest training points |
| 3 | Naive Bayes | Probabilistic, assumes feature independence |
| 4 | Decision Trees | Axis-aligned splits, interpretable |
| 5 | Random Forest | Ensemble of trees with feature/sample randomness |
| 6 | Gradient Boosting / XGBoost | Sequential tree building, corrects errors |

---

## ⚡ Quick Reference Tables

### Algorithm Comparison
| Model | Handles Non-linearity | Interpretable | Speed (train) | Needs Scaling |
|-------|----------------------|---------------|----------------|---------------|
| Logistic Regression | No (linear) | ✅ | Fast | ✅ |
| KNN | Yes | Moderate | N/A (lazy) | ✅ |
| Naive Bayes | Limited | ✅ | Very fast | ❌ |
| Decision Tree | Yes | ✅ | Fast | ❌ |
| Random Forest | Yes | Moderate | Medium | ❌ |
| XGBoost | Yes | Limited | Medium | ❌ |

### When to Use Each
| Situation | Try First |
|-----------|----------|
| Baseline / interpretability needed | Logistic Regression |
| Text/spam classification | Naive Bayes |
| Small dataset, quick prototype | Decision Tree |
| Best accuracy, tabular data | XGBoost / Random Forest |
| No feature scaling available | Tree-based models |

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report
np.random.seed(42)

X, y = make_classification(n_samples=1000, n_features=10, n_informative=6,
                            n_redundant=2, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr); X_te_s = scaler.transform(X_te)
print(f"Dataset: {X.shape}, class balance: {np.bincount(y)}")

---
## 1. Logistic Regression

### What this cell does:
Logistic regression fits a linear model then passes through the sigmoid to produce probabilities. The decision boundary is linear. Coefficients are interpretable as log-odds. C parameter is the inverse of regularization strength (higher C = less regularization).

In [ ]:
lr = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
lr.fit(X_tr_s, y_tr)
pred_lr = lr.predict(X_te_s)
print(f"Logistic Regression accuracy: {accuracy_score(y_te, pred_lr):.4f}")
print("\nClassification Report:")
print(classification_report(y_te, pred_lr))
print("\nProbability outputs (first 5):", lr.predict_proba(X_te_s)[:5].round(3))

---
## 2. KNN

### What this cell does:
KNN classifies by majority vote from k nearest neighbors. Demonstrates how k affects the decision boundary — small k = complex (overfits), large k = smooth (underfits). Must scale features because KNN uses Euclidean distance.

In [ ]:
for k in [1, 5, 15, 50]:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_tr_s, y_tr)
    acc = accuracy_score(y_te, knn.predict(X_te_s))
    print(f"KNN k={k:2d}: accuracy={acc:.4f}")

---
## 3 & 4. Naive Bayes & Decision Tree

### What this cell does:
Naive Bayes applies Bayes' theorem assuming features are independent given class — fast but naive. Decision tree recursively splits on best features using Gini/entropy. The tree is visualized for the first 3 levels.

In [ ]:
nb = GaussianNB()
nb.fit(X_tr_s, y_tr)
print(f"Naive Bayes accuracy: {accuracy_score(y_te, nb.predict(X_te_s)):.4f}")

dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_tr, y_tr)  # trees don't need scaling
print(f"Decision Tree accuracy: {accuracy_score(y_te, dt.predict(X_te)):.4f}")

fig, ax = plt.subplots(figsize=(15, 6))
plot_tree(dt, max_depth=3, filled=True, ax=ax, fontsize=7)
ax.set_title('Decision Tree (first 3 levels)')
plt.tight_layout(); plt.show()

---
## 5 & 6. Random Forest & Gradient Boosting / XGBoost

### What this cell does:
Random Forest builds trees on random subsets of data and features, averages predictions — reduces variance. Gradient Boosting builds trees sequentially, each correcting the previous — reduces bias. Feature importances show which features matter most.

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_tr, y_tr)
print(f"Random Forest accuracy: {accuracy_score(y_te, rf.predict(X_te)):.4f}")

gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
gb.fit(X_tr, y_tr)
print(f"Gradient Boosting accuracy: {accuracy_score(y_te, gb.predict(X_te)):.4f}")

# Feature importances from RF
importances = pd.Series(rf.feature_importances_, index=[f'f{i}' for i in range(X.shape[1])]).sort_values(ascending=True)
importances.plot.barh(figsize=(8, 4), title='Random Forest Feature Importances')
plt.tight_layout(); plt.show()

# All models summary
print("\n=== All Models Summary ===")
all_models = [
    ('Logistic Reg', lr, X_te_s), ('KNN(k=5)', KNeighborsClassifier(5).fit(X_tr_s,y_tr), X_te_s),
    ('Naive Bayes', nb, X_te_s), ('Decision Tree', dt, X_te), ('Random Forest', rf, X_te), ('Grad Boost', gb, X_te)
]
for name, model, Xtest in all_models:
    print(f"  {name:<18}: {accuracy_score(y_te, model.predict(Xtest)):.4f}")